In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import math
import os

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD  = (0.2023, 0.1994, 0.2010)
CLASSES = ['plane','car','bird','cat','deer','dog','frog','horse','ship','truck']


In [2]:

def get_loaders(batch_size=128):
    tr = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
    ])
    te = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
    ])
    train_ds = torchvision.datasets.CIFAR10('./data', train=True,  download=True, transform=tr)
    test_ds  = torchvision.datasets.CIFAR10('./data', train=False, download=True, transform=te)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=2, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    return train_loader, test_loader

class PatchEmbed(nn.Module):
    def __init__(self, patch_size=4, embed_dim=256):
        super().__init__()
        self.patch_size = patch_size
        self.proj = nn.Conv2d(3, embed_dim, kernel_size=patch_size, stride=patch_size)
        self.num_patches = (32 // patch_size) ** 2

    def forward(self, x):
        return self.proj(x).flatten(2).transpose(1, 2)

class Attention(nn.Module):
    def __init__(self, dim, heads=8):
        super().__init__()
        self.heads = heads
        self.head_dim = dim // heads
        self.scale = self.head_dim ** -0.5
        self.qkv  = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)

    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.heads, self.head_dim).permute(2,0,3,1,4)
        q, k, v = qkv.unbind(0)
        attn = (q @ k.transpose(-2,-1)) * self.scale
        attn = attn.softmax(dim=-1)
        x = (attn @ v).transpose(1,2).reshape(B, N, C)
        return self.proj(x)

class TransformerBlock(nn.Module):
    def __init__(self, dim, heads=8, mlp_ratio=4.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn  = Attention(dim, heads)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp   = nn.Sequential(
            nn.Linear(dim, int(dim * mlp_ratio)),
            nn.GELU(),
            nn.Linear(int(dim * mlp_ratio), dim),
        )

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x

class ViTPretrainer(nn.Module):
    def __init__(self, patch_size=4, embed_dim=256, depth=2, heads=8, num_classes=10):
        super().__init__()
        self.patch_embed = PatchEmbed(patch_size, embed_dim)
        num_patches = self.patch_embed.num_patches
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        self.blocks = nn.ModuleList([TransformerBlock(embed_dim, heads) for _ in range(depth)])
        self.norm   = nn.LayerNorm(embed_dim)
        self.head   = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        B = x.shape[0]
        x = self.patch_embed(x)
        cls = self.cls_token.expand(B, -1, -1)
        x   = torch.cat([cls, x], dim=1) + self.pos_embed
        for blk in self.blocks:
            x = blk(x)
        x = self.norm(x)
        return self.head(x[:, 0])

    def get_patch_features(self, x):
        B = x.shape[0]
        x = self.patch_embed(x)
        x = x + self.pos_embed[:, 1:, :]
        for blk in self.blocks:
            cls_pad = torch.zeros(B, 1, x.shape[-1], device=x.device)
            x_with_cls = torch.cat([cls_pad, x], dim=1)
            x_with_cls = blk(x_with_cls)
            x = x_with_cls[:, 1:, :]
        return self.norm(x)


    



In [3]:
class SpatialGraph(nn.Module):
    def __init__(self, grid_size=8, alpha=0.5):
        super().__init__()
        self.grid_size = grid_size
        self.alpha     = alpha
        self.register_buffer('spatial_adj', self._build_spatial())

    def _build_spatial(self):
        N   = self.grid_size ** 2
        adj = torch.zeros(N, N)
        for i in range(self.grid_size):
            for j in range(self.grid_size):
                src = i * self.grid_size + j
                for di in [-1, 0, 1]:
                    for dj in [-1, 0, 1]:
                        if di == 0 and dj == 0:
                            continue
                        ni, nj = i + di, j + dj
                        if 0 <= ni < self.grid_size and 0 <= nj < self.grid_size:
                            dst = ni * self.grid_size + nj
                            dist = math.sqrt(di**2 + dj**2)
                            adj[src, dst] = math.exp(-dist**2 / 2.0)
        return adj

    def forward(self, features):
        B, N, D = features.shape
        feat_norm = F.normalize(features, dim=-1)
        feat_sim  = torch.bmm(feat_norm, feat_norm.transpose(1, 2))
        spatial   = self.spatial_adj.unsqueeze(0).expand(B, -1, -1)
        mask      = (spatial > 0).float()
        combined  = self.alpha * spatial + (1 - self.alpha) * feat_sim
        return combined * mask

def train_one_epoch(model, loader, opt, criterion):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        opt.zero_grad()
        out  = model(x)
        loss = criterion(out, y)
        loss.backward()
        opt.step()
        total_loss += loss.item()
        correct    += out.argmax(1).eq(y).sum().item()
        total      += y.size(0)
    return total_loss / len(loader), 100.0 * correct / total

@torch.no_grad()
def eval_one_epoch(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        out  = model(x)
        loss = criterion(out, y)
        total_loss += loss.item()
        correct    += out.argmax(1).eq(y).sum().item()
        total      += y.size(0)
    return total_loss / len(loader), 100.0 * correct / total

def train_vit(epochs=30, patch_size=4, lr=1e-3, batch_size=128):
    print(f"\n{'='*50}")
    print(f"ViT Pretraining | patch={patch_size} | device={DEVICE}")
    print(f"{'='*50}")
    train_loader, test_loader = get_loaders(batch_size)
    model     = ViTPretrainer(patch_size=patch_size).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    opt       = optim.AdamW(model.parameters(), lr=lr, weight_decay=0.05)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    history   = {'train_loss':[], 'train_acc':[], 'test_loss':[], 'test_acc':[]}
    for ep in range(1, epochs+1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, opt, criterion)
        te_loss, te_acc = eval_one_epoch(model,  test_loader,  criterion)
        scheduler.step()
        history['train_loss'].append(tr_loss)
        history['train_acc'].append(tr_acc)
        history['test_loss'].append(te_loss)
        history['test_acc'].append(te_acc)
        print(f"Ep {ep:03d}/{epochs} | tr_loss={tr_loss:.4f} tr_acc={tr_acc:.2f}% | te_loss={te_loss:.4f} te_acc={te_acc:.2f}%")
    ckpt_name = f'vit_patch{patch_size}.pth'
    torch.save(model.state_dict(), ckpt_name)
    print(f"Saved: {ckpt_name}")
    return model,history

In [4]:
def sanity_check_patches(patch_size=4):
    print(f"\n--- Sanity Check: Patches (patch_size={patch_size}) ---")
    _, test_loader = get_loaders(64)
    imgs, labels   = next(iter(test_loader))
    img  = imgs[0]
    grid = 32 // patch_size
    num  = grid ** 2
    embed = PatchEmbed(patch_size, 256)
    with torch.no_grad():
        patches = embed(img.unsqueeze(0))
    print(f"Image shape:   {img.shape}")
    print(f"Grid size:     {grid}x{grid}")
    print(f"Num patches:   {num}")
    print(f"Patch shape:   {patches.shape}")
    mean = (0.4914, 0.4822, 0.4465)
    std  = (0.2023, 0.1994, 0.2010)
    img_show = img.clone()
    for c in range(3):
        img_show[c] = img_show[c] * std[c] + mean[c]
    img_show = img_show.permute(1, 2, 0).numpy().clip(0, 1)
    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    axes[0].imshow(img_show)
    axes[0].set_title(f'Original | label={CLASSES[labels[0]]}')
    axes[0].axis('off')
    overlay = img_show.copy()
    for i in range(1, grid):
        overlay[i*patch_size, :, :] = [1, 0, 0]
        overlay[:, i*patch_size, :] = [1, 0, 0]
    axes[1].imshow(overlay)
    axes[1].set_title(f'{grid}x{grid} Grid ({num} patches)')
    axes[1].axis('off')
    plt.tight_layout()
    plt.savefig(f'sanity_patches_{patch_size}.png', dpi=120)
    plt.close()
    print(f"Saved: sanity_patches_{patch_size}.png")

def sanity_check_graph(model, patch_size=4, alpha=0.5):
    print(f"\n--- Sanity Check: Spatial Graph (patch_size={patch_size}, alpha={alpha}) ---")
    _, test_loader = get_loaders(64)
    imgs, labels   = next(iter(test_loader))
    imgs = imgs.to(DEVICE)
    model.eval()
    with torch.no_grad():
        features = model.get_patch_features(imgs[:4])
    grid   = 32 // patch_size
    graph  = SpatialGraph(grid_size=grid, alpha=alpha).to(DEVICE)
    adj    = graph(features)
    print(f"Feature shape: {features.shape}")
    print(f"Adj shape:     {adj.shape}")
    print(f"Edges per node (spatial): {(graph.spatial_adj > 0).float().sum(1).mean().item():.1f}")
    print(f"Adj min/max:   {adj.min().item():.4f} / {adj.max().item():.4f}")
    print(f"Adj mean (nonzero): {adj[adj > 0].mean().item():.4f}")
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    axes[0].imshow(graph.spatial_adj.cpu().numpy(), cmap='hot')
    axes[0].set_title('Spatial Adjacency (fixed)')
    axes[0].set_xlabel('Node')
    axes[0].set_ylabel('Node')
    plt.colorbar(axes[0].images[0], ax=axes[0])
    feat_norm = F.normalize(features[0], dim=-1)
    feat_sim  = torch.mm(feat_norm, feat_norm.t()).cpu().numpy()
    axes[1].imshow(feat_sim, cmap='viridis')
    axes[1].set_title('Feature Similarity')
    axes[1].set_xlabel('Node')
    axes[1].set_ylabel('Node')
    plt.colorbar(axes[1].images[0], ax=axes[1])
    axes[2].imshow(adj[0].cpu().numpy(), cmap='plasma')
    axes[2].set_title(f'Combined (alpha={alpha})')
    axes[2].set_xlabel('Node')
    axes[2].set_ylabel('Node')
    plt.colorbar(axes[2].images[0], ax=axes[2])
    plt.tight_layout()
    plt.savefig(f'sanity_graph_p{patch_size}_a{int(alpha*10)}.png', dpi=120)
    plt.close()
    print(f"Saved: sanity_graph_p{patch_size}_a{int(alpha*10)}.png")
    return features, adj

def sanity_check_alpha(model, patch_sizes=[4, 8], alphas=[0.3, 0.5, 0.7]):
    print(f"\n--- Sanity Check: Alpha Comparison ---")
    _, test_loader = get_loaders(64)
    imgs, _  = next(iter(test_loader))
    imgs     = imgs.to(DEVICE)
    model.eval()
    fig, axes = plt.subplots(len(patch_sizes), len(alphas), figsize=(4*len(alphas), 4*len(patch_sizes)))
    for pi, ps in enumerate(patch_sizes):
        with torch.no_grad():
            tmp_model = ViTPretrainer(patch_size=ps).to(DEVICE)
            features  = tmp_model.get_patch_features(imgs[:1])
        grid = 32 // ps
        for ai, alpha in enumerate(alphas):
            graph = SpatialGraph(grid_size=grid, alpha=alpha).to(DEVICE)
            adj   = graph(features)
            ax    = axes[pi][ai] if len(patch_sizes) > 1 else axes[ai]
            ax.imshow(adj[0].cpu().numpy(), cmap='plasma')
            ax.set_title(f'patch={ps}, α={alpha}')
            ax.set_xlabel('Node')
            ax.set_ylabel('Node')
            nonzero_mean = adj[0][adj[0] > 0].mean().item()
            print(f"  patch={ps} alpha={alpha}: nonzero_mean={nonzero_mean:.4f} nodes={grid**2}")
    plt.tight_layout()
    plt.savefig('sanity_alpha_comparison.png', dpi=120)
    plt.close()
    print(f"Saved: sanity_alpha_comparison.png")

def plot_training(history, patch_size):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
    ax1.plot(history['train_loss'], label='train')
    ax1.plot(history['test_loss'],  label='test')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title(f'Loss (patch={patch_size})')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    ax2.plot(history['train_acc'], label='train')
    ax2.plot(history['test_acc'],  label='test')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy (%)')
    ax2.set_title(f'Accuracy (patch={patch_size})')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'training_patch{patch_size}.png', dpi=120)
    plt.close()
    print(f"Saved: training_patch{patch_size}.png")



In [5]:
if __name__ == '__main__':
    os.makedirs('outputs', exist_ok=True)

    print("STEP 1: Sanity check patches (before training)")
    sanity_check_patches(patch_size=4)
    sanity_check_patches(patch_size=8)

    print("\nSTEP 2: Train ViT with patch_size=4 (8x8 grid, 64 patches)")
    model_4, history_4 = train_vit(epochs=30, patch_size=4)
    plot_training(history_4, patch_size=4)

    print("\nSTEP 3: Train ViT with patch_size=8 (4x4 grid, 16 patches)")
    model_8, history_8 = train_vit(epochs=30, patch_size=8)
    plot_training(history_8, patch_size=8)

    print("\nSTEP 4: Sanity check spatial graphs")
    sanity_check_graph(model_4, patch_size=4, alpha=0.5)
    sanity_check_graph(model_8, patch_size=8, alpha=0.5)

    print("\nSTEP 5: Alpha comparison across both grids")
    sanity_check_alpha(model_4, patch_sizes=[4, 8], alphas=[0.3, 0.5, 0.7,0.6])

    print("\nAll steps complete.")
    print("Files saved:")
    for f in sorted(os.listdir('.')):
        if f.endswith('.png') or f.endswith('.pth'):
            print(f"  {f}")

STEP 1: Sanity check patches (before training)

--- Sanity Check: Patches (patch_size=4) ---


100%|██████████| 170M/170M [00:08<00:00, 20.6MB/s] 


Image shape:   torch.Size([3, 32, 32])
Grid size:     8x8
Num patches:   64
Patch shape:   torch.Size([1, 64, 256])
Saved: sanity_patches_4.png

--- Sanity Check: Patches (patch_size=8) ---
Image shape:   torch.Size([3, 32, 32])
Grid size:     4x4
Num patches:   16
Patch shape:   torch.Size([1, 16, 256])
Saved: sanity_patches_8.png

STEP 2: Train ViT with patch_size=4 (8x8 grid, 64 patches)

ViT Pretraining | patch=4 | device=cuda
Ep 001/40 | tr_loss=1.7692 tr_acc=34.09% | te_loss=1.6233 te_acc=41.36%
Ep 002/40 | tr_loss=1.4578 tr_acc=46.71% | te_loss=1.3421 te_acc=52.14%
Ep 003/40 | tr_loss=1.3533 tr_acc=50.57% | te_loss=1.2812 te_acc=52.52%
Ep 004/40 | tr_loss=1.2859 tr_acc=53.31% | te_loss=1.1903 te_acc=56.86%
Ep 005/40 | tr_loss=1.2298 tr_acc=55.19% | te_loss=1.1698 te_acc=57.29%
Ep 006/40 | tr_loss=1.1887 tr_acc=56.94% | te_loss=1.1353 te_acc=58.80%
Ep 007/40 | tr_loss=1.1607 tr_acc=58.03% | te_loss=1.0961 te_acc=60.34%
Ep 008/40 | tr_loss=1.1207 tr_acc=59.46% | te_loss=1.0692 te_